# Drum RM – copaci fără marcare (model care caută după nume)

Un model gata antrenat (YOLO-World) caută în poze: **trunchi de copac**, **groapă**, **indicator rutier**. Nu trebuie marcat nimic.

1. **Runtime → Change runtime type → T4 GPU → Save**.
2. **Runtime → Run all**.
3. Când scrie *„Aștept pozele”*: în stânga apasă iconița **dosar 📁**, apoi iconița **Upload** (săgeata în sus) din panou și alege pozele cu copaci. (Sau trage-le cu mouse-ul în panou.)
4. Caietul le găsește singur și arată pozele cu trunchiurile găsite. La final se descarcă **drum_v2.onnx**.

In [ ]:
!pip -q install ultralytics
import torch; print('Placa video:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NU – alege T4 GPU din Runtime')

In [ ]:
# Modelul care caută după nume. Ordinea claselor e cea din programul Drum: groapa, copac, semn
from ultralytics import YOLOWorld
CLASE = ['pothole', 'tree trunk', 'traffic sign']
# varianta mică (s): model de ~50 MB, rapid pe procesor; 'l' e mai precisă dar are 177 MB
m = YOLOWorld('yolov8s-worldv2.pt')
m.set_classes(CLASE)
print('model gata')

In [ ]:
# Pozele: din panoul de fișiere din stânga (📁 → Upload). Nu se folosește butonul din pagină (se blochează pe unele rețele).
import glob, os, time
EXT = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
def poze(): return sorted(p for p in glob.glob('/content/*') + glob.glob('/content/poze/*') if p.lower().endswith(EXT) and '/rez_' not in p)
if not poze():
    print('Aștept pozele: în stânga apasă 📁, apoi Upload (săgeata în sus) și alege pozele cu copaci…')
t0 = time.time(); n = -1
while time.time() - t0 < 900:
    p = poze()
    if p and len(p) == n: break          # s-au terminat de încărcat
    n = len(p); time.sleep(5)
POZE = poze(); print(len(POZE), 'poze găsite:', [os.path.basename(p) for p in POZE])

In [ ]:
# Pragul de încredere: mai mic = mai mulți copaci găsiți (și mai multe greșeli)
PRAG = 0.10
import glob, cv2
from IPython.display import Image, display
for p in POZE:
    r = m.predict(p, conf=PRAG, imgsz=1280, verbose=False)[0]
    img = cv2.imread(p); n = 0
    for (x0, y0, x1, y1), c, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        col = [(50, 50, 230), (60, 200, 60), (230, 120, 40)][int(c)]
        cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), col, 2)
        if int(c) == 1: cv2.circle(img, (int((x0 + x1) / 2), int(y1)), 6, (0, 0, 255), -1); n += 1
    out = '/content/rez_' + os.path.basename(p).rsplit('.', 1)[0] + '.jpg'; cv2.imwrite(out, img)
    print(os.path.basename(p), ':', n, 'trunchiuri'); display(Image(out, width=900))

In [ ]:
# Modelul pentru programul Drum (merge pe orice calculator, fără placă video)
import shutil, os, zipfile
from google.colab import files
f = m.export(format='onnx', imgsz=640, simplify=True)
shutil.copy(f, '/content/drum_v2.onnx')
z = '/content/drum_v2.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as a: a.write('/content/drum_v2.onnx', 'drum_v2.onnx')
mb = os.path.getsize(z) / 2**20; print(f'drum_v2.zip: {mb:.1f} MB')
if mb <= 25:
    files.download(z)
else:
    # bucăți de câte 25 MB (limita pentru trimis): drum_v2.zip.001, .002, ...
    d = open(z, 'rb').read(); n = 0
    for i in range(0, len(d), 25 * 2**20):
        n += 1; p = f'/content/drum_v2.zip.{n:03d}'; open(p, 'wb').write(d[i:i + 25 * 2**20]); files.download(p)
    print(n, 'bucăți – trimite-le pe toate')